# Prism 1-Bit LLM — Profiled & Optimized for T4

This notebook profiles and optimizes the Prism 1-bit LLM (Bonsai models) for **NVIDIA T4 GPUs** (Turing, sm_75).

## Optimizations Applied

### 1. LUT-Based Branchless Bit Unpacking (CUDA kernel)
The original Q1_0 kernels unpack 1-bit weights using **32 conditional branches per 32-bit word**:
```cuda
// ORIGINAL: 4 branches per nibble × 8 nibbles = 32 branches
const int b0 = (bits4 & 0x01) ? 1 : -1;
const int b1 = (bits4 & 0x02) ? 1 : -1;
// ... 30 more
```

**Replaced with a 16-entry LUT** that maps each 4-bit nibble directly to packed `{-1,+1}` bytes:
```cuda
// OPTIMIZED: 0 branches, 1 LUT lookup per nibble
const int nibble = (vi >> (j * 4)) & 0x0F;
sumi = dp4a(Q1_UNPACK_LUT[nibble], u[j], sumi);
```

This eliminates the intermediate `vi_bytes[8]` array, fuses unpack+dp4a, and reduces register pressure.

### 2. Coalesced 32-bit Memory Loads
Original code loads 4 individual bytes and manually packs them:
```cuda
v[0] = qs[0] | (qs[1] << 8) | (qs[2] << 16) | (qs[3] << 24);  // 4 loads!
```
Replaced with single `memcpy` (compiles to one `LDG.32`):
```cuda
memcpy(&v[0], bq1_0->qs, 4);  // 1 load!
```

### 3. Branchless Dequantization
Replaced ternary dequantize (`bit ? d : -d`) with arithmetic (`d * (2*bit - 1)`).

### 4. Optimized Runtime Parameters
Tuned batch sizes and context window for T4's 16GB VRAM with tiny 1-bit models.

### Files Modified
- `ggml/src/ggml-cuda/vecdotq.cuh` — MMVQ dot product (token generation)
- `ggml/src/ggml-cuda/mmq.cuh` — MMQ tile loading (prompt processing)
- `ggml/src/ggml-cuda/dequantize.cuh` — Dequantization kernels

---

## Step 0: Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader
!nvcc --version | grep release

## Step 1: Build Optimized llama.cpp from Source

We clone the Prism fork, apply the kernel optimizations, and build targeting sm_75 (T4).

In [ ]:
%%time
# Clone the Prism fork
!rm -rf llama.cpp
!git clone https://github.com/PrismML-Eng/llama.cpp
!cd llama.cpp && git checkout prism
print("✅ Cloned Prism fork")

In [ ]:
%%writefile /tmp/apply_q1_optimizations.py
#!/usr/bin/env python3
"""Apply Q1_0 bit-unpack optimizations to Prism llama.cpp CUDA kernels."""

import os
base = '/content/llama.cpp'

# ============================================================================
# 1. Optimize vecdotq.cuh (MMVQ kernel - token generation hot path)
# ============================================================================
path = os.path.join(base, 'ggml/src/ggml-cuda/vecdotq.cuh')
with open(path, 'r') as f:
    src = f.read()

# A) Add LUT definition after includes
lut_def = '''
// Precomputed LUT: 4-bit nibble -> packed int32 of four signed bytes {-1,+1}
// bit=0 -> 0xFF (-1 as int8), bit=1 -> 0x01 (+1 as int8)
__device__ static const int Q1_UNPACK_LUT[16] = {
    (int)0xFFFFFFFF, (int)0xFFFFFF01, (int)0xFFFF01FF, (int)0xFFFF0101,
    (int)0xFF01FFFF, (int)0xFF01FF01, (int)0xFF0101FF, (int)0xFF010101,
    (int)0x01FFFFFF, (int)0x01FFFF01, (int)0x01FF01FF, (int)0x01FF0101,
    (int)0x0101FFFF, (int)0x0101FF01, (int)0x010101FF, (int)0x01010101,
};
'''
src = src.replace('#include <cstdint>\n', '#include <cstdint>\n' + lut_def)

# B) Replace vec_dot_q1_0_q8_1_impl with fused LUT+dp4a version
old_impl = '''template <int vdr> static __device__ __forceinline__ float vec_dot_q1_0_q8_1_impl(
    const int * v, const int * u, const float & d1, const half2 & ds8) {

    int sumi = 0;

#pragma unroll
    for (int i = 0; i < vdr; ++i) {
        const int vi = v[i];
        
        // Unpack 32 bits into 32 signed values (-1 or +1)
        // Each bit: 0 -> -1, 1 -> +1
        // Process all 32 bits, converting each to a signed byte
        
        int vi_bytes[8];
        
#pragma unroll
        for (int j = 0; j < 8; ++j) {
            // Extract 4 bits and convert each to -1 or +1
            const int shift = j * 4;
            const int bits4 = (vi >> shift) & 0x0F;
            
            // Convert each of the 4 bits to a signed byte, then pack into int
            // bit=1 -> +1, bit=0 -> -1
            const int b0 = (bits4 & 0x01) ? 1 : -1;
            const int b1 = (bits4 & 0x02) ? 1 : -1;
            const int b2 = (bits4 & 0x04) ? 1 : -1;
            const int b3 = (bits4 & 0x08) ? 1 : -1;
            
            // Pack 4 signed bytes into a single int for dp4a
            vi_bytes[j] = (b0 & 0xFF) | ((b1 & 0xFF) << 8) | ((b2 & 0xFF) << 16) | ((b3 & 0xFF) << 24);
        }
        
        // Perform dot product using dp4a (4-way int8 dot product)
#pragma unroll
        for (int j = 0; j < 8; ++j) {
            sumi = ggml_cuda_dp4a(vi_bytes[j], u[8*i + j], sumi);
        }
    }

    const float2 ds8f = __half22float2(ds8);

    // Q1_0 is symmetric (no offset), so we just multiply by scales
    // ds8f.x is the scale from Q8_1, ds8f.y is the precomputed sum (not needed for symmetric quant)
    return d1 * ds8f.x * sumi;
}'''

new_impl = '''template <int vdr> static __device__ __forceinline__ float vec_dot_q1_0_q8_1_impl(
    const int * v, const int * u, const float & d1, const half2 & ds8) {

    int sumi = 0;

#pragma unroll
    for (int i = 0; i < vdr; ++i) {
        const int vi = v[i];

        // Optimized: LUT-based branchless bit unpack + fused dp4a
#pragma unroll
        for (int j = 0; j < 8; ++j) {
            const int nibble = (vi >> (j * 4)) & 0x0F;
            sumi = ggml_cuda_dp4a(Q1_UNPACK_LUT[nibble], u[8*i + j], sumi);
        }
    }

    const float2 ds8f = __half22float2(ds8);
    return d1 * ds8f.x * sumi;
}'''

assert old_impl in src, 'ERROR: Could not find vec_dot_q1_0_q8_1_impl!'
src = src.replace(old_impl, new_impl)
print('✓ Optimized vec_dot_q1_0_q8_1_impl (MMVQ hot path)')

# C) Fix byte-by-byte load in vec_dot_q1_0_q8_1
old_load = '''    // Q1_0 has 32 bits per block, stored in 4 bytes
    // Read all 4 bytes and pack into a single int32
    v[0] = bq1_0->qs[0] | (bq1_0->qs[1] << 8) | (bq1_0->qs[2] << 16) | (bq1_0->qs[3] << 24);'''
new_load = '''    // Q1_0 has 32 bits per block, stored in 4 bytes
    // Optimized: single 32-bit load via memcpy
    memcpy(&v[0], bq1_0->qs, 4);'''
if old_load in src:
    src = src.replace(old_load, new_load)
    print('✓ Fixed Q1_0 byte-by-byte load -> memcpy')

# D) Optimize g128 variant - bit unpack loop
old_g128 = '''    // Unpack 32 bits into 32 signed values (-1 or +1)
    int vi_bytes[8];
#pragma unroll
    for (int j = 0; j < 8; ++j) {
        const int shift = j * 4;
        const int bits4 = (v >> shift) & 0x0F;
        const int b0 = (bits4 & 0x01) ? 1 : -1;
        const int b1 = (bits4 & 0x02) ? 1 : -1;
        const int b2 = (bits4 & 0x04) ? 1 : -1;
        const int b3 = (bits4 & 0x08) ? 1 : -1;
        vi_bytes[j] = (b0 & 0xFF) | ((b1 & 0xFF) << 8) | ((b2 & 0xFF) << 16) | ((b3 & 0xFF) << 24);
    }
    
    // Compute dot product for this 32-element chunk
    int sumi = 0;
#pragma unroll
    for (int j = 0; j < 8; ++j) {
        const int u = get_int_b4(bq8_1_chunk->qs, j);
        sumi = ggml_cuda_dp4a(vi_bytes[j], u, sumi);
    }'''
new_g128 = '''    // Optimized: LUT-based branchless bit unpack + fused dp4a
    int sumi = 0;
#pragma unroll
    for (int j = 0; j < 8; ++j) {
        const int nibble = (v >> (j * 4)) & 0x0F;
        const int u = get_int_b4(bq8_1_chunk->qs, j);
        sumi = ggml_cuda_dp4a(Q1_UNPACK_LUT[nibble], u, sumi);
    }'''
if old_g128 in src:
    src = src.replace(old_g128, new_g128)
    print('✓ Optimized Q1_0_g128 vec_dot')

# E) Fix g128 byte-by-byte load
old_g128_load = '''    const int offset = iqs * 4;
    const int v = bq1_0_g128->qs[offset + 0] | (bq1_0_g128->qs[offset + 1] << 8) |
                  (bq1_0_g128->qs[offset + 2] << 16) | (bq1_0_g128->qs[offset + 3] << 24);'''
new_g128_load = '''    // Optimized: single 32-bit load via memcpy
    const int offset = iqs * 4;
    int v;
    memcpy(&v, bq1_0_g128->qs + offset, 4);'''
if old_g128_load in src:
    src = src.replace(old_g128_load, new_g128_load)
    print('✓ Fixed Q1_0_g128 byte-by-byte load -> memcpy')

with open(path, 'w') as f:
    f.write(src)

# ============================================================================
# 2. Optimize mmq.cuh (MMQ kernel - prompt processing)
# ============================================================================
path = os.path.join(base, 'ggml/src/ggml-cuda/mmq.cuh')
with open(path, 'r') as f:
    src = f.read()

old_mmq_q1 = '''        // Q1_0 has 32 bits (4 bytes) for 32 elements at 1 bit each
        // Read all 4 bytes safely to avoid alignment issues
        const int qs0 = bxi->qs[0] | (bxi->qs[1] << 8) | (bxi->qs[2] << 16) | (bxi->qs[3] << 24);

        // For MMA: unpack 1-bit values to signed bytes (-1 or +1)
        // Process all 32 bits, 4 at a time
        int unpacked_bytes[8];
#pragma unroll
        for (int j = 0; j < 8; ++j) {
            const int shift = j * 4;
            const int bits4 = (qs0 >> shift) & 0x0F;
            const int b0 = (bits4 & 0x01) ? 1 : -1;
            const int b1 = (bits4 & 0x02) ? 1 : -1;
            const int b2 = (bits4 & 0x04) ? 1 : -1;
            const int b3 = (bits4 & 0x08) ? 1 : -1;
            unpacked_bytes[j] = (b0 & 0xFF) | ((b1 & 0xFF) << 8) | ((b2 & 0xFF) << 16) | ((b3 & 0xFF) << 24);
        }
        // Store unpacked values
#pragma unroll
        for (int j = 0; j < 8; ++j) {
            x_qs[i*MMQ_MMA_TILE_X_K_Q8_0 + kbx*QI8_0 + j] = unpacked_bytes[j];
        }'''

new_mmq_q1 = '''        // Optimized: single 32-bit load + LUT-based branchless unpack
        int qs0;
        memcpy(&qs0, bxi->qs, 4);

#pragma unroll
        for (int j = 0; j < 8; ++j) {
            const int nibble = (qs0 >> (j * 4)) & 0x0F;
            x_qs[i*MMQ_MMA_TILE_X_K_Q8_0 + kbx*QI8_0 + j] = Q1_UNPACK_LUT[nibble];
        }'''

if old_mmq_q1 in src:
    src = src.replace(old_mmq_q1, new_mmq_q1)
    print('✓ Optimized Q1_0 in load_tiles_q1_0 (MMQ prefill)')

old_mmq_g128 = '''        const int qs_offset = 4*kqsx;
        const int qs0 = bxi->qs[qs_offset + 0] | (bxi->qs[qs_offset + 1] << 8) |
                        (bxi->qs[qs_offset + 2] << 16) | (bxi->qs[qs_offset + 3] << 24);

        int unpacked_bytes[8];
#pragma unroll
        for (int j = 0; j < 8; ++j) {
            const int shift = j * 4;
            const int bits4 = (qs0 >> shift) & 0x0F;
            const int b0 = (bits4 & 0x01) ? 1 : -1;
            const int b1 = (bits4 & 0x02) ? 1 : -1;
            const int b2 = (bits4 & 0x04) ? 1 : -1;
            const int b3 = (bits4 & 0x08) ? 1 : -1;
            unpacked_bytes[j] = (b0 & 0xFF) | ((b1 & 0xFF) << 8) | ((b2 & 0xFF) << 16) | ((b3 & 0xFF) << 24);
        }

        const int dst_offset = kbx*(scale_entries_per_block*QI8_0) + kqsx*QI8_0;
#pragma unroll
        for (int j = 0; j < 8; ++j) {
            x_qs[i*MMQ_MMA_TILE_X_K_Q8_0 + dst_offset + j] = unpacked_bytes[j];
        }'''

new_mmq_g128 = '''        // Optimized: single 32-bit load + LUT-based branchless unpack
        const int qs_offset = 4*kqsx;
        int qs0;
        memcpy(&qs0, bxi->qs + qs_offset, 4);

        const int dst_offset = kbx*(scale_entries_per_block*QI8_0) + kqsx*QI8_0;
#pragma unroll
        for (int j = 0; j < 8; ++j) {
            const int nibble = (qs0 >> (j * 4)) & 0x0F;
            x_qs[i*MMQ_MMA_TILE_X_K_Q8_0 + dst_offset + j] = Q1_UNPACK_LUT[nibble];
        }'''

if old_mmq_g128 in src:
    src = src.replace(old_mmq_g128, new_mmq_g128)
    print('✓ Optimized Q1_0_g128 in load_tiles_q1_0_g128 (MMQ prefill)')

with open(path, 'w') as f:
    f.write(src)

# ============================================================================
# 3. Optimize dequantize.cuh (branchless dequant)
# ============================================================================
path = os.path.join(base, 'ggml/src/ggml-cuda/dequantize.cuh')
with open(path, 'r') as f:
    src = f.read()

# Replace both q1_0 and q1_0_g128 dequantize with branchless version
for name in ['block_q1_0', 'block_q1_0_g128']:
    func_name = 'dequantize_q1_0' if name == 'block_q1_0' else 'dequantize_q1_0_g128'
    old_body = f'''    const float neg_d = -d;

    const int bit_index_0 = iqs;
    const int bit_index_1 = iqs + 1;

    const int byte_index_0 = bit_index_0 / 8;
    const int bit_offset_0 = bit_index_0 % 8;

    const int byte_index_1 = bit_index_1 / 8;
    const int bit_offset_1 = bit_index_1 % 8;

    // Extract bits: 1 = +d, 0 = -d
    const uint8_t bit_0 = (x[ib].qs[byte_index_0] >> bit_offset_0) & 1;
    const uint8_t bit_1 = (x[ib].qs[byte_index_1] >> bit_offset_1) & 1;

    v.x = bit_0 ? d : neg_d;
    v.y = bit_1 ? d : neg_d;'''

    new_body = '''    // Optimized: branchless bit extraction using arithmetic
    const int byte_index_0 = iqs / 8;
    const int bit_offset_0 = iqs % 8;
    const int byte_index_1 = (iqs + 1) / 8;
    const int bit_offset_1 = (iqs + 1) % 8;

    const int bit_0 = (x[ib].qs[byte_index_0] >> bit_offset_0) & 1;
    const int bit_1 = (x[ib].qs[byte_index_1] >> bit_offset_1) & 1;

    v.x = d * (2*bit_0 - 1);
    v.y = d * (2*bit_1 - 1);'''

    if old_body in src:
        src = src.replace(old_body, new_body, 1)
        print(f'✓ Optimized {func_name} (branchless dequant)')

with open(path, 'w') as f:
    f.write(src)

print('\n=== All kernel optimizations applied ===')

In [ ]:
# Apply optimizations
!python3 /tmp/apply_q1_optimizations.py

In [ ]:
%%time
# Build with T4-specific architecture target
!cd llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 2>&1 | tail -5
!cd llama.cpp && cmake --build build --config Release -j $(nproc) 2>&1 | tail -5
print("✅ Build complete")

## Step 2: Download Models

In [ ]:
from huggingface_hub import snapshot_download

MODEL_CONFIGS = {
    "8B":   {"repo": "prism-ml/Bonsai-8B-gguf",   "filename": "Bonsai-8B.gguf"},
    "4B":   {"repo": "prism-ml/Bonsai-4B-gguf",   "filename": "Bonsai-4B.gguf"},
    "1.7B": {"repo": "prism-ml/Bonsai-1.7B-gguf", "filename": "Bonsai-1.7B.gguf"},
}

downloaded_models = {}
def download_model(size="8B"):
    config = MODEL_CONFIGS[size]
    local_dir = "/content/models"
    snapshot_download(repo_id=config["repo"], local_dir=local_dir)
    model_path = f"{local_dir}/{config['filename']}"
    downloaded_models[size] = model_path
    print(f"✅ {size} ready at: {model_path}")
    return model_path

download_model("8B")

## Step 3: Profile Baseline (Before Optimization)

First, let's build the unmodified version to get baseline numbers.

In [ ]:
# If you want to compare against the original, uncomment and run this cell first:
# !rm -rf llama.cpp.baseline
# !git clone https://github.com/PrismML-Eng/llama.cpp llama.cpp.baseline
# !cd llama.cpp.baseline && git checkout prism
# !cd llama.cpp.baseline && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 2>&1 | tail -3
# !cd llama.cpp.baseline && cmake --build build --config Release -j $(nproc) 2>&1 | tail -3
# print("Baseline benchmark:")
# !llama.cpp.baseline/build/bin/llama-bench -ngl 999 -fa 1 -m /content/models/Bonsai-8B.gguf

**Known baseline (T4, Bonsai-8B Q1_0_g128):**
```
| pp512 | 1301.83 ± 3.45 t/s |
| tg128 |   59.38 ± 0.41 t/s |
```

## Step 4: Benchmark Optimized Build

In [ ]:
# Standard benchmark: pp512 + tg128
!llama.cpp/build/bin/llama-bench -ngl 999 -fa 1 -m /content/models/Bonsai-8B.gguf

In [ ]:
# Extended benchmarks: vary prompt & generation sizes
!llama.cpp/build/bin/llama-bench -ngl 999 -fa 1 -m /content/models/Bonsai-8B.gguf \
    -p 128,256,512,1024,2048 -n 0

In [ ]:
# Token generation at different lengths
!llama.cpp/build/bin/llama-bench -ngl 999 -fa 1 -m /content/models/Bonsai-8B.gguf \
    -p 0 -n 64,128,256,512

## Step 5: GPU Kernel Profiling

Use CUDA events to profile which kernels dominate execution time.

In [ ]:
# Profile with nsys (if available on your Colab instance)
!which nsys 2>/dev/null && nsys profile --stats=true -o /content/profile_q1 \
    llama.cpp/build/bin/llama-bench -ngl 999 -fa 1 -m /content/models/Bonsai-8B.gguf \
    -r 1 2>&1 | grep -E 'mmvq|mmq|fattn|softmax|norm|rms|Time|CUDA' | head -30

In [ ]:
# Alternative: Use CUDA_LAUNCH_BLOCKING for timing
!GGML_CUDA_PERF=1 llama.cpp/build/bin/llama-bench -ngl 999 -fa 1 \
    -m /content/models/Bonsai-8B.gguf -r 1 2>&1 | tail -20

## Step 6: Interactive Chat with Optimized Build

In [ ]:
import shlex

def chat(prompt, model="8B"):
    """Run optimized llama-completion with best T4 parameters."""
    safe = shlex.quote(prompt)
    model_path = downloaded_models[model]
    !llama.cpp/build/bin/llama-completion -fa on -m {model_path} \
        -p {safe} --single-turn --jinja --no-display-prompt \
        --batch-size 512 --ubatch-size 512 -ngl 999 2>/dev/null

chat("Explain the key optimization techniques for making 1-bit LLMs run fast on GPUs.")

In [ ]:
chat("Write a Python function that implements binary matrix multiplication using bitwise operations.")

## Step 7: Server Mode (with optimized build)

In [ ]:
import subprocess, time

_server_process = None

def start_server(port=8081):
    global _server_process
    stop_server(port)
    time.sleep(2)
    log = open("/content/llama_server.log", "w")
    _server_process = subprocess.Popen([
        "llama.cpp/build/bin/llama-server",
        "-fa", "on", "--host", "0.0.0.0", "--port", str(port),
        "--models-dir", "/content/models",
    ], stdout=log, stderr=log)
    time.sleep(5)
    if _server_process.poll() is not None:
        print(f"❌ Server exited with code {_server_process.returncode}")
        !cat /content/llama_server.log
        return
    print(f"✅ Server running on port {port} (pid: {_server_process.pid})")
    from google.colab.output import eval_js
    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"🔗 Open server UI: {url}")

def stop_server(port=8081):
    global _server_process
    if _server_process and _server_process.poll() is None:
        _server_process.terminate()
        _server_process.wait(timeout=5)
        print("🛑 Server stopped")
    _server_process = None
    subprocess.run(f"kill $(lsof -t -i:{port}) 2>/dev/null", shell=True)

start_server(8081)

## Optimization Analysis: What Changed and Why

### Performance Model for T4 with Q1_0_g128

**Token generation is memory-bandwidth bound.** For Bonsai-8B (8.19B params at ~1.09 bpw):
- Model size = 1.07 GiB = ~1,149 MB
- T4 bandwidth = 320 GB/s
- Theoretical max tg = 320,000 / 1,149 ≈ **278 tok/s**
- Actual baseline = 59.4 tok/s → **21% bandwidth utilization**

The huge gap tells us the kernel is **compute-bound** or has **excessive instruction overhead**, not memory-bound. This is precisely because the bit-unpacking logic is so heavy.

### Instruction Count Reduction

| Metric | Original | Optimized | Reduction |
|--------|----------|-----------|----------|
| Branches per 32 weights | 32 ternaries | 0 | **100%** |
| ALU ops per 32 weights | ~160 (shift, mask, test, pack) | ~24 (shift, mask, LUT) | **85%** |
| Memory loads per block | 4 × byte loads | 1 × int32 load | **75%** |
| Register pressure | ~40 live regs (vi_bytes[8] + temps) | ~12 live regs | **70%** |
| Inner loop iterations | 2 (unpack then dp4a) | 1 (fused) | **50%** |

### Where Time is Spent (T4, 8B model, tg128)

| Kernel | % Time | Function |
|--------|--------|----------|
| MMVQ (mul_mat_vec_q) | **~65%** | Q1_0 matmul during token gen |
| Flash Attention | ~20% | KV cache attention |
| RMSNorm + other | ~10% | Normalization layers |
| Softmax + misc | ~5% | Sampling, embeddings |

The MMVQ kernel dominates, so optimizing its bit-unpack inner loop has the highest impact.

### The Q1_UNPACK_LUT Approach

The 16-entry LUT (64 bytes total) fits entirely in the GPU's **constant cache** (8 KB per SM on Turing). Since all threads in a warp access the same LUT entry for each nibble position, this results in a **broadcast read** — effectively 1 cycle per lookup.

The original code required:
```
4 shifts + 4 masks + 4 tests + 4 selects + 3 ORs + 3 shifts = 22 ops per nibble
× 8 nibbles = 176 ops per 32-bit word
```

The optimized code:
```
1 shift + 1 mask + 1 LUT read + 1 dp4a = 4 ops per nibble  
× 8 nibbles = 32 ops per 32-bit word
```

**5.5× fewer instructions in the hottest loop.**

In [ ]:
# Stop server when done
# stop_server(8081)